<p style="text-align: center; margin-bottom: 0; font-size: 24px;">EAE1106 - Métodos Computacionais para Economia</p>
<p style="text-align: center; margin-bottom: 0;">Faculdade de Economia, Administração, Contabilidade e Atuária<br>Universidade de São Paulo</p>
<p style="text-align: center; margin-top: 5px;">1º semestre de 2026<br><br>Prof. Arthur Viaro</p>

# Aula 21 — Manipulação de Dados com Pandas (Continuação)

**Conteúdo**

- [Mudando o Formato dos DataFrames](#reshape)
- [Trabalhando com Múltiplos DataFrames](#multiplos)
- [Trabalhando com Datetimes](#datetimes)

In [ ]:
import pandas as pd
import numpy as np

## Mudando o Formato dos DataFrames <a id="reshape"></a>

**Tidy data** (dados arrumados) significa alinhar a estrutura do DataFrame ao seu significado. Dados tidy seguem três princípios:

- Cada variável é uma coluna;
- Cada observação é uma linha;
- Cada tipo de unidade observacional é uma tabela separada.

Muitas vezes, precisamos reorganizar ("dar reshape") um DataFrame para deixá-lo tidy ou para facilitar análises e visualizações.

![](https://r4ds.had.co.nz/images/tidy-1.png)

### Melt e Pivot

Os métodos `.melt()`, `.pivot()` e `.pivot_table()` do Pandas ajudam a reorganizar ("reshape") DataFrames:

- `.melt()`: transforma dados largos em longos.
- `.pivot()`: transforma dados longos em largos.
- `.pivot_table()`: igual ao `.pivot()`, mas permite múltiplos índices e agregações quando há duplicatas.

![](https://github.com/gadenbuie/tidyexplain/raw/main/images/static/png/original-dfs-tidy.png)

O exemplo abaixo mostra quantos cursos diferentes professores ministraram em diferentes anos. Se a pergunta for algo como: "O número de cursos ministrados varia conforme o ano?", a tabela abaixo não estaria em formato tidy, pois há múltiplas observações de cursos por ano em uma mesma linha (ou seja, há dados de 2023, 2024 e 2025 em uma única linha):

In [ ]:
df = pd.DataFrame({
    "Nome": ["Professor A", "Professor B", "Professor C", "Professor D", "Professor E"],
    "2023": [1, 3, 4, 5, 3],
    "2024": [2, 4, 3, 2, 1],
    "2025": [5, 2, 4, 4, 3]
})

df

Vamos deixar o DataFrame tidy usando o método `.melt()`. O argumento mais importante é `id_vars`, que indica qual coluna será usada como "identificador".

In [ ]:
df_melt = df.melt(id_vars = "Nome",
                  var_name = "Ano",
                  value_name = "Cursos")
df_melt

O argumento `value_vars` permite escolher **quais** colunas serão "derretidas". Se omitido, todas as colunas que não forem `id_vars` são incluídas. No exemplo abaixo, excluímos 2023 — somente 2024 e 2025 aparecem no resultado:

In [ ]:
df.melt(id_vars = "Nome",
        value_vars = ["2024", "2025"],
        var_name = "Ano",
        value_name = "Cursos")

Para fazer o caminho inverso — de dados longos de volta para largos — usamos `.pivot()`. É basicamente o inverso do `.melt()`. Precisamos informar qual coluna vira o índice (`index`), qual vira os nomes das novas colunas (`columns`) e qual fornece os valores (`values`):

In [ ]:
df_pivot = df_melt.pivot(index = "Nome",
                         columns = "Ano",
                         values = "Cursos")
print(df_pivot)

Perceba que o Pandas criou um índice hierárquico com o nome `"Nome"` e preservou o rótulo `"Ano"` no topo das colunas. Para recuperar a aparência original do DataFrame (sem esses rótulos), use `reset_index()` para transformar o índice em coluna:

In [ ]:
df_pivot = df_pivot.reset_index()
df_pivot.columns.name = None
print(df_pivot)

O método `.pivot()` geralmente resolve o nosso problema, mas não vai funcionar se:

- Quisermos usar múltiplos índices;
- Tivermos rótulos duplicados de índice/coluna.

Nesses casos, teremos que usar o `.pivot_table()`. Consulte a [documentação](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html#pivot-tables) para mais detalhes.

In [ ]:
df = pd.DataFrame({
    "Nome": ["Professor A", "Professor A", "Professor B", "Professor B"],
    "Departamento": ["Economia", "Administração", "Economia", "Administração"],
    "2023": [1, 2, 3, 1],
    "2024": [2, 3, 4, 2],
    "2025": [5, 1, 2, 2]
}).melt(id_vars=["Nome", "Departamento"], var_name = "Ano", value_name = "Cursos")

df

No exemplo acima, cada professor aparece duas vezes em `Nome` (uma linha para Economia, outra para Administração). Ao tentar pivotar usando só `Nome`, o Pandas encontra dois valores para a mesma combinação (Nome, Ano) e não sabe qual usar — gerando um `ValueError: Index contains duplicate entries, cannot reshape`:

In [ ]:
df.pivot(index = "Nome",
         columns = "Ano",
         values = "Cursos")

Nesse caso, usamos `.pivot_table()`. Ele aplica uma **função de agregação** quando há duplicatas na mesma célula. Abaixo somamos os cursos de todos os departamentos por professor e por ano:

In [ ]:
df.pivot_table(
    index = "Nome",
    columns = "Ano",
    values = "Cursos",
    aggfunc = "sum")

Se quisermos manter os números por departamento, podemos especificar tanto `Nome` quanto `Departamento` como múltiplos índices:

In [ ]:
df_pivot = df.pivot_table(
    index = ["Nome", "Departamento"],
    columns = "Ano",
    values = "Cursos"
)

df_pivot

O resultado acima é um DataFrame com multi-índice, ou seja, um índice hierárquico. Se algum dia precisar usar esse recurso, consulte a [documentação](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html#pivot-tables) do `pivot_table()` para mais detalhes.

In [ ]:
df_pivot.xs('Professor A', level = "Nome")

## Trabalhando com Múltiplos DataFrames <a id="multiplos"></a>

Frequentemente, precisamos trabalhar com múltiplos DataFrames que precisam ser combinados ou "colados". Os métodos `df.merge()` e `pd.concat()` são as principais ferramentas para isso. A documentação do Pandas é bastante clara sobre essas funções, mas elas são simples de usar na prática.

### Combinando DataFrames com `pd.concat()`

Podemos usar `pd.concat()` para juntar DataFrames de duas formas:

- **Verticalmente**: quando eles têm as mesmas colunas (as linhas são empilhadas uma embaixo da outra).
- **Horizontalmente**: quando eles têm os mesmos índices/linhas (as colunas são "coladas" lado a lado).

Essas operações são muito úteis para consolidar dados que vêm de diferentes arquivos, períodos ou fontes.

In [ ]:
df1 = pd.DataFrame({
    'A': [1, 3, 5],
    'B': [2, 4, 6]
})

df1

In [ ]:
df2 = pd.DataFrame({
    'A': [7, 9, 11],
    'C': [8, 10, 12]
})

df2

In [ ]:
# Combinando verticalmente (axis = 0)
pd.concat((df1, df2), axis = 0)

Perceba que os índices foram simplesmente empilhados — df1 tinha índice 0,1,2 e df2 também, então o resultado repete esses valores. Para renumerar o índice de forma contínua, use `ignore_index=True`:

In [ ]:
pd.concat((df1, df2), axis = 0, ignore_index = True)

Use `axis=1` para colar DataFrames lado a lado (horizontalmente). Útil quando dois DataFrames descrevem os mesmos registros mas com colunas diferentes:

In [ ]:
pd.concat((df1, df2), axis = 1)

In [ ]:
pd.concat((df1, df2), axis = 1, ignore_index = True)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['df1', 'df2'])
#pd.concat([df1, df2], axis=1, keys=['df1', 'df2'])['df2']['A']

Não estamos limitados a apenas dois DataFrames — podemos concatenar quantos quisermos:

In [ ]:
pd.concat((df1, df2, df1, df2), axis = 0, ignore_index = True)

### Combinando DataFrames com `pd.merge()`

O método `pd.merge()` permite "juntar" DataFrames usando diferentes regras (assim como no SQL, se você já conhece). Podemos usar `df.merge()` para unir DataFrames com base em colunas-chave compartilhadas. Os principais tipos de junção são:

- "inner join" (junção interna)
- "left join" (junção à esquerda)
- "right join" (junção à direita)
- "outer join" (junção externa)

![Fonte: Livro “R for Data Science (2e)” de “Hadley Wickham and Garrett Grolemund”.](https://r4ds.hadley.nz/diagrams/join/venn.png)

Veja este ótimo [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) para mais detalhes.

Vamos começar apresentando uma representação visual das junções. Nestes exemplos, usaremos uma única coluna-chave chamada `chave` e uma única coluna de valor (`val_x` em df1 e `val_y` em df2), mas as ideias se generalizam para múltiplas chaves e múltiplos valores.

![Fonte: Livro "R for Data Science (2e)" de "Hadley Wickham and Garrett Grolemund".](https://r4ds.hadley.nz/diagrams/join/setup.png)

In [ ]:
df1 = pd.DataFrame({
    'chave': [1, 2, 3],
    'val_x': ['x1', 'x2', 'x3']
})

df1

In [ ]:
df2 = pd.DataFrame({
    'chave': [1, 2, 4],
    'val_y': ['y1', 'y2', 'y3']
})

df2

#### Inner Join

Uma "junção interna" (inner join) retorna apenas as linhas cuja chave aparece em **ambos** os DataFrames. Registros que existem em apenas um dos lados são descartados.

No exemplo: df1 tem chaves [1, 2, 3] e df2 tem [1, 2, 4] — o resultado conterá apenas as chaves 1 e 2, que são comuns.

![Fonte: Livro "R for Data Science (2e)" de "Hadley Wickham and Garrett Grolemund".](https://r4ds.hadley.nz/diagrams/join/inner.png)

In [ ]:
pd.merge(df1, df2, how = "inner", on = "chave")

# pd.merge(
    # left = df1,
    # right = df2,
    # how = "inner",
    # left_on = "chave",
    # right_on = "chave"
# )

#### Left Join

Uma "junção à esquerda" (left join) retorna **todas** as linhas de df1, complementadas pelas colunas de df2 onde houver correspondência. Quando não há correspondência no lado direito, as colunas de df2 ficam como `NaN`.

Em outras palavras, nenhuma linha do DataFrame da esquerda (df1) é perdida — o da direita é que pode ter lacunas.

![Fonte: Livro "R for Data Science (2e)" de "Hadley Wickham and Garrett Grolemund".](https://r4ds.hadley.nz/diagrams/join/left.png)

In [ ]:
pd.merge(df1, df2, how = "left", on = "chave")

#### Right Join

Uma "junção à direita" (right join) é o espelho do left join: retorna **todas** as linhas de df2, complementadas pelas colunas de df1 onde houver correspondência. Quando não há correspondência no lado esquerdo, as colunas de df1 ficam como `NaN`.

Em outras palavras, nenhuma linha do DataFrame da direita (df2) é perdida — o da esquerda é que pode ter lacunas.

![Fonte: Livro "R for Data Science (2e)" de "Hadley Wickham and Garrett Grolemund".](https://r4ds.hadley.nz/diagrams/join/right.png)

In [ ]:
pd.merge(df1, df2, how = "right", on = "chave")

#### Outer Join

Uma "junção completa" (outer join ou full join) é a mais inclusiva: mantém **todas** as linhas de ambos os DataFrames, independentemente de haver correspondência. Onde não há correspondência, os valores do lado ausente ficam como `NaN`.

No exemplo: df1 tem a chave 3 (sem par em df2) e df2 tem a chave 4 (sem par em df1) — ambas aparecem no resultado, com `NaN` nas colunas do lado que não as possui.

![Fonte: Livro "R for Data Science (2e)" de "Hadley Wickham and Garrett Grolemund".](https://r4ds.hadley.nz/diagrams/join/full.png)

In [ ]:
pd.merge(df1, df2, how = "outer", on = "chave")

Há muitas maneiras de especificar a chave de junção: pelo índice (`left_index=True`/`right_index=True`), por colunas com nomes diferentes (`left_on=`/`right_on=`), etc.

O argumento `indicator=True` adiciona uma coluna `_merge` ao resultado, informando para cada linha se ela veio do DataFrame esquerdo (`left_only`), direito (`right_only`) ou de ambos (`both`). Isso é muito útil para auditar a qualidade de uma junção:

In [ ]:
pd.merge(df1, df2, how = "outer", on = "chave", indicator = True)

## Trabalhando com Datetimes <a id="datetimes"></a>

Grande parte dos dados econômicos são séries temporais: IPCA mensal, PIB trimestral, Ibovespa diário, taxa de câmbio intradiária. Para trabalhar bem com esses dados, precisamos que o Python entenda o que é uma **data de verdade** — não apenas um número ou uma string.

O desafio é que datas têm semântica própria: meses têm comprimentos diferentes, existem anos bissextos, feriados, fusos horários, etc. Por isso, Python, NumPy e Pandas oferecem estruturas especializadas para lidar com datas e horas. Cada uma adiciona uma camada de funcionalidade em cima da anterior:

- **`datetime` (Python nativo)**: objetos de data e hora para valores individuais
- **NumPy `datetime64`**: operações vetorizadas sobre arrays de datas
- **Pandas `Timestamp` / `DatetimeIndex`**: indexação temporal, reamostragem, fuso horário, e muito mais

### O módulo `datetime` do Python

O Python nativo oferece o módulo `datetime`, que permite criar objetos de data e hora com semântica correta — ou seja, o Python "sabe" que `2026-02-28 + 1 dia = 2026-03-01`, por exemplo.

| Classe      | Função             |
| ----------- | ------------------ |
| `date`      | apenas data        |
| `time`      | apenas horário     |
| `datetime`  | data + hora        |
| `timedelta` | diferença de tempo |
| `timezone`  | fuso horário       |

In [ ]:
from datetime import datetime, timedelta

In [ ]:
date = datetime(year = 2026, month = 5, day = 21, hour = 19, minute = 30)
date

Com um objeto `datetime`, podemos converter qualquer componente de volta para string usando `strftime()` (*string format time* — o inverso de `strptime`). O resultado é uma string formatada conforme os [códigos de formatação](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes):

In [ ]:
date = datetime.strptime("May 21 2026, 13:54", "%B %d %Y, %H:%M")
date

#datetime.strptime("21/05/2025", "%d/%m/%Y")

Com um objeto `datetime`, podemos extrair qualquer componente usando `strftime()` (*string format time*, o inverso de `strptime`):

In [ ]:
date.strftime("%Y-%m-%d")

In [ ]:
print(f"Ano: {date.strftime('%Y')}")
print(f"Mês: {date.strftime('%B')}")
print(f"Dia: {date.strftime('%d')}")
print(f"Data: {date.strftime('%Y-%m-%d')}")
print(f"Dia da semana: {date.strftime('%A')}")
print(f"Dia do ano: {date.strftime('%j')}")
print(f"Período do dia: {date.strftime('%p')}")

Operações aritméticas com datas usam `timedelta`, que representa uma **duração** (diferença de tempo). Podemos somar ou subtrair `timedelta` de um `datetime` para deslocar uma data no tempo:

In [ ]:
print(date + timedelta(days=7))
print(date - timedelta(days=2, hours=2))

O módulo `datetime` funciona bem para datas individuais, mas é lento para operar sobre **milhares de datas de uma vez** — o que é muito comum em séries temporais econômicas. O NumPy resolve isso com o tipo `datetime64`, que armazena datas em arrays compactos e permite operações vetorizadas eficientes:

In [ ]:
# Criando datas
np.datetime64('2025-05-21')

In [ ]:
# Arrays de datas
dates = np.array(["2020-05-21", "2020-06-10"], dtype = "datetime64")
dates

Para gerar sequências regulares de datas, use `np.arange()`. O parâmetro `dtype='datetime64[unidade]'` define a granularidade: `D` = dia, `M` = mês, `Y` = ano:

In [ ]:
dates = np.arange("2020-06", "2020-12", dtype = 'datetime64[M]')
dates

Com arrays `datetime64`, operações vetorizadas funcionam sobre todos os elementos de uma vez — sem laços. O exemplo abaixo soma 2 meses a cada data do array simultaneamente. Consulte todas as unidades disponíveis na [documentação do NumPy](https://numpy.org/doc/stable/reference/arrays.datetime.html#datetime-units):

In [ ]:
dates + np.timedelta64(2, 'M')

Embora o NumPy torne as operações com datas mais eficientes, ele ainda carece de funcionalidades importantes para análise de dados — como reamostragem, indexação parcial por período ou suporte a fusos horários. É aí que o **Pandas** entra.

O Pandas unifica e estende as funcionalidades do `datetime` e do NumPy em quatro objetos principais:

| Objeto          | Equivalente no NumPy | O que representa                         | Exemplo                |
| --------------- | -------------------- | ---------------------------------------- | ---------------------- |
| `Timestamp`     | `np.datetime64`      | Um instante específico no tempo          | `2026-05-18 10:30`     |
| `DatetimeIndex` | —                    | Um conjunto indexado de datas e horários | Série temporal diária  |
| `Timedelta`     | `np.timedelta64`     | Uma duração ou diferença entre datas     | `3 dias`, `2 horas`    |
| `Period`        | —                    | Um intervalo fixo de tempo               | `maio/2026`, `Q1-2026` |

### Criando Datetimes com Pandas

**Do zero**

As três funções mais usadas para criar datetimes no Pandas são:

- `pd.Timestamp()` — cria **um momento no tempo** (como `np.datetime64`, mas com muito mais funcionalidades)
- `pd.Period()` — cria **um intervalo de tempo** (ex: `2020-01` = o mês inteiro de janeiro de 2020)
- `pd.date_range()` / `pd.period_range()` — criam **sequências regulares** de timestamps ou períodos

`pd.Timestamp()` aceita strings, argumentos nomeados (`year`, `month`, `day`, etc.) ou um objeto `datetime` nativo:

In [ ]:
# Convertido de string
print(pd.Timestamp('2025-05-21'))

# Dados passados diretamente
print(pd.Timestamp(year=2025, month=5, day=21))

# A partir de objeto datetime
print(pd.Timestamp(datetime(year=2025, month=5, day=21)))

A diferença entre `Timestamp` e `Period` é fundamental:

- **`Timestamp`** = um instante exato no tempo (como um "clique" no relógio — um ponto sem duração)
- **`Period`** = um intervalo de tempo com começo e fim (o dia `2005-07-09` começa à meia-noite e termina às 23:59:59)

In [ ]:
span = pd.Period('2005-05-21')
print(span)
print(span.start_time)
print(span.end_time)

Um `Timestamp` pode ser comparado aos limites de um `Period` para verificar se ele "cai dentro" do intervalo:

In [ ]:
point = pd.Timestamp('2005-05-21 12:00')
span = pd.Period('2005-05-21')
print(f"Ponto: {point}")
print(f"Intervalo: {span}")
print(f"Ponto está no intervalo? {span.start_time < point < span.end_time}")

Na prática, raramente trabalhamos com datas isoladas — o mais comum é ter uma **sequência regular de datas** como índice de um DataFrame (ex: todos os dias úteis de 2020, todos os meses de 2015 a 2024, etc.). Para isso usamos:

- `pd.date_range()`: retorna um `DatetimeIndex` de instantes, ou seja, uma lsita de `Timestamps` (datas/horários espeecíficos)
- `pd.period_range()`: retorna um `PeriodIndex` de intervalos

O parâmetro mais importante é `freq`:

| Frequência | Significado         |
| ---------- | ------------------- |
| `'D'`      | diário              |
| `'B'`      | dias úteis          |
| `'W'`      | semanal             |
| `'MS'`     | início do mês       |
| `'ME'`     | fim do mês          |
| `'QS'`     | início do trimestre |
| `'QE'`     | fim do trimestre    |
| `'YS'`     | início do ano       |
| `'YE'`     | fim do ano          |
| `'h'`      | hora                |
| `'min'`    | minuto              |

In [ ]:
pd.date_range('2026-01', '2026-04', freq = 'MS')

In [ ]:
pd.period_range('2026-01', '2026-04', freq = 'M')

`Timedelta` representa uma duração e pode ser somado a um `DatetimeIndex` inteiro de uma vez — útil, por exemplo, para ajustar o fuso horário de toda uma série ou para criar janelas deslizantes:

In [ ]:
pd.date_range('2020-01-01', '2020-12-01', freq='D') + pd.Timedelta('4 days')

Datas ausentes no Pandas são representadas por `NaT` (*Not a Time*) — o equivalente temporal do `NaN` para números. Aparece, por exemplo, quando uma string de data não pode ser convertida, ou quando há lacunas em uma série temporal. Ao criar um `Timestamp` a partir de `NaT`, o resultado é `NaT`:

In [ ]:
pd.Timestamp(pd.NaT)

**Convertendo dados já existentes**

Na prática, as datas costumam chegar como strings — vindas de CSVs, APIs ou bancos de dados. `pd.to_datetime()` converte essas strings para `datetime`, reconhecendo automaticamente os formatos mais comuns:

In [ ]:
string_dates = ['May 21, 2020', 'August 1, 2020', 'August 28, 2020']
string_dates

In [ ]:
pd.to_datetime(string_dates)

Para formatos ambíguos ou não convencionais (ex: `"2020 21 May"`), `pd.to_datetime()` pode não reconhecer automaticamente. Nesses casos, use o argumento `format=` com os [códigos de formatação do Python](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes) para descrever explicitamente o padrão da string:

In [ ]:
string_dates = ['2020-21,May', '2020-1,August', '2020-28,August']
pd.to_datetime(string_dates, format="%Y-%d,%B")

**Criando datas a partir de um dicionário**

Quando ano, mês e dia estão em **colunas separadas** de um DataFrame (situação comum em planilhas), podemos montar datas passando um dicionário diretamente para `pd.to_datetime()`. O resultado é uma `Series` de `Timestamps`:

In [ ]:
df = pd.DataFrame({
    'ano': [2025, 2025, 2025],
    'mes': [7, 8, 8],
    'dia': [9, 1, 28],
    'valor': [5.21, 5.18, 5.08]
})

df

In [ ]:
datas = pd.to_datetime({
    'year': df['ano'],
    'month': df['mes'],
    'day': df['dia']
})

datas

In [ ]:
df['data'] = datas
df

Nesse momento, as datas ainda são apenas valores comuns associados às linhas do DataFrame. Quando queremos usar essas datas como índice temporal de um DataFrame, convertemos a estrutura para um DatetimeIndex (índice temporal do DataFrame) com `pd.Index()`.

Assim, as datas deixam de ser apenas valores de uma coluna e passam a funcionar como um índice temporal, facilitando operações típicas de séries temporais, como filtros por data, reamostragem e ordenação cronológica.

In [ ]:
df.index = pd.Index(datas)
df

**Lendo de um arquivo externo**

Na prática, os dados chegam de CSVs, Excel ou bancos de dados — e o Pandas costuma importar colunas de data como `object` (strings). Vamos usar a série histórica do **Ibovespa** como exemplo prático de como lidar com isso:

In [ ]:
df = pd.read_excel('ibovespa.xlsx')
df

Note que a coluna Data já está no tipo `datetime64`, mas o DataFrame ainda usa um índice numérico padrão (`RangeIndex`).

Para aproveitar totalmente as funcionalidades temporais do Pandas, normalmente transformamos a coluna de datas em um `DatetimeIndex`:

Para habilitar a indexação temporal, primeiro garantimos que a coluna de datas esteja no tipo `datetime64` usando `pd.to_datetime()`. Em seguida, definimos essa coluna como índice do DataFrame com `.set_index()`:

In [ ]:
df['Data'] = pd.to_datetime(df['Data'])
df = df.set_index('Data')
df

Melhor ainda: podemos fazer isso tudo em uma única linha, passando `parse_dates=True` e `index_col='Data'` diretamente em `read_excel()`. O Pandas converte e define o índice automaticamente ao ler o arquivo:

In [ ]:
df = pd.read_excel('ibovespa.xlsx', parse_dates = True, index_col = 'Data')
df

In [ ]:
df.index

### Indexação com Datetimes

Uma das grandes vantagens de ter um `DatetimeIndex` é a **indexação parcial por string**: você pode selecionar dados usando apenas parte da data — um ano, um mês, um dia — sem precisar de filtros booleanos explícitos. O Pandas interpreta a string e seleciona automaticamente todos os registros correspondentes.

Abaixo, o Ibovespa com `DatetimeIndex`:

In [ ]:
df

Selecionar todos os registros de um mês inteiro — sem filtro booleano:

In [ ]:
df.loc['2019-05']

Para um dia específico:

In [ ]:
df.loc['2010-05-21']

Para um intervalo de datas, use fatiamento com `.loc[]`. Diferente do fatiamento por posição (que exclui o último elemento), aqui **ambas as extremidades são incluídas**:

In [ ]:
df.loc['2010-05-21':'2010-10-21']

### Manipulando Datetimes

**Decomposição**

Com um `DatetimeIndex`, é possível extrair qualquer componente temporal como um array numérico — útil para criar variáveis, agrupar dados por período ou analisar sazonalidade. Os atributos mais usados são `.year`, `.month`, `.day`, `.weekday` (0 = segunda-feira), `.hour`, `.quarter`, etc.:

In [ ]:
df.index.year

In [ ]:
df.index.month

In [ ]:
df.index.weekday

Além dos atributos numéricos, há métodos que retornam o **nome** do componente como string — útil para rótulos em gráficos e tabelas (ex: `"Monday"`, `"January"`):

In [ ]:
df.index.day_name()

In [ ]:
df.index.month_name()

**Atenção (o acessor `.dt`)**: os atributos acima funcionam diretamente sobre um `DatetimeIndex`. Se você tiver uma **coluna** `datetime` dentro de uma `Series`, precisa acessar os atributos pelo acessor `.dt` — caso contrário, o Pandas lança um `AttributeError`:

```python
s.year      # AttributeError: não funciona em Series
s.dt.year   # ✓ correto
```

In [ ]:
s = pd.Series(pd.date_range('2011-12-29', '2011-12-31'))
s.year  # gera erro

In [ ]:
s.dt.year  # funciona

### Reamostragem e Agregação

**Reamostrar** significa mudar a **frequência** de uma série temporal: transformar dados diários em semanais, mensais em anuais, etc. O método `.resample()` funciona como um `.groupby()` temporal — agrupa os dados por intervalo de tempo e aplica uma função de agregação.

A sintaxe é `df.resample("frequência").função()`, onde a frequência segue os [aliases do Pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#dateoffset-objects):

| Alias | Frequência |
|---|---|
| `"D"` | Dia |
| `"W"` | Semana |
| `"ME"` | Fim do mês |
| `"QE"` | Fim do trimestre |
| `"YE"` | Fim do ano |

O objeto `Resampler` funciona exatamente como um `GroupBy` — não retorna dados por si só, precisa de uma função de agregação:

```python
# GroupBy:  df.groupby("coluna").mean()
# Resample: df.resample("W").mean()
```

#### Exemplo: Dados Climáticos do INMET

O [INMET (Instituto Nacional de Meteorologia)](https://portal.inmet.gov.br/) disponibiliza séries históricas de estações automáticas em todo o Brasil. Os dados são coletados de hora em hora e incluem variáveis como temperatura, umidade relativa e precipitação acumulada.

Este é um caso de uso clássico para **reamostragem**: os dados chegam com frequência horária, mas análises climáticas geralmente precisam de médias diárias ou totais mensais.

**Lendo o arquivo**

Os CSVs do INMET têm 8 linhas de cabeçalho com metadados da estação (nome, coordenadas, altitude, período) antes dos dados tabulares — por isso usamos `skiprows=8`. O separador de colunas é `;` e os decimais são vírgulas:

In [ ]:
df = pd.read_csv(
    'inmet.csv',
    encoding = 'iso-8859-1',
    decimal = ',',
    delimiter = ';',
    skiprows = 8
)

df

**Construindo o índice temporal**

Os dados têm data e hora em colunas separadas — `Data` no formato `DD/MM/AAAA` e `Hora UTC` no formato `HHMM UTC`. Precisamos combiná-las em um único `datetime` para usá-lo como índice:

In [ ]:
# Remove " UTC"
df['Hora UTC'] = df['Hora UTC'].str.replace(' UTC', '', regex=False)

# Combina data + hora
df['datetime'] = pd.to_datetime(
    df['Data'] + ' ' + df['Hora UTC'],
    format='%d/%m/%Y %H%M'
)

df[['Data', 'Hora UTC', 'datetime']].head()

O DataFrame agora tem uma nova coluna `datetime`, mas `Data` e `Hora UTC` ainda estão presentes. Veja o estado atual — em seguida removemos as colunas redundantes e usamos `datetime` como índice:

In [ ]:
df

In [ ]:
# Definir coluna Timestamp como índice
df = df.set_index('datetime')
df = df.drop(columns = ['Data', 'Hora UTC'])
df

Com o DataFrame limpo e indexado temporalmente, podemos inspecionar as estatísticas descritivas e confirmar que o índice é de fato um `DatetimeIndex` com frequência horária:

In [ ]:
df.index

In [ ]:
df.describe().round(2)

**Reamostragem diária**

Os dados chegaram com frequência horária, mas análises climáticas geralmente trabalham com granularidade diária. `resample("D").mean()` agrupa todas as leituras de cada dia e calcula a **média diária** de cada variável:

In [ ]:
dfr = df.resample("D").mean().round(2)
dfr

**Reamostragem mensal com agregações diferentes por coluna**

A mesma função de agregação nem sempre faz sentido para todas as variáveis. Para precipitação, o indicador correto é o **total mensal acumulado** (`sum`); para temperatura e umidade, a **média mensal** (`mean`) é mais informativa. Usamos `.agg()` com um dicionário para atribuir uma função diferente a cada coluna:

In [ ]:
dfr = df.resample("ME").agg({
    'Precipitacao': 'sum',
    'Temperatura': 'mean',
    'Umidade': 'mean'
}).round(2)

dfr['Mês'] = dfr.index.month_name()
dfr